# QuickSwathScanner: every place Biomass has imaged

Orlaith Doyle (Ní Dhúill), ESA-ESTEC Biomass Intern, with supervisor Björn Rommen

Biomass revisits the same swath locations again and again for its tomographic repeat passes, so the Level-1a catalogue on ESA MAAP holds hundreds of thousands of granules but only a few thousand genuinely different places. Before I could test anything over the ocean I needed to know where the mission had actually looked, without downloading or processing every acquisition.

This notebook scans the whole `BiomassLevel1a` STAC collection once, keeps one record per unique swath location (`grid:code`), and works out how much of each footprint is ocean using Natural Earth land polygons. Each swath then gets called ocean, coastal or land and plotted on a world map. On my full run that was 223,315 granules down to 5,167 unique swath locations, about 2.3 % of the catalogue.

A full scan takes something like 8 to 11 hours and the API falls over now and then, so the scan is built to checkpoint as it goes and pick up where it left off. You can stop it, restart the kernel, come back the next day and just run the cells again.

**How to run it**

1. Section 1, set the paths
2. Section 2, log in and load any checkpoint (run this at the start of every session)
3. Section 3, the scan itself, rerun it as many times as it takes to reach the end
4. Everything after that reads the CSVs straight off disk, so it can be run any time, even halfway through a scan

**Output files** (all in `BASE_DIR`)

| File | What's in it |
|---|---|
| `scan_state.json` | the resume checkpoint: count so far, grid codes seen, last datetime, boundary ids |
| `all_swaths.csv` | one row per unique `grid_code`: footprint (WKT), ocean fraction, orbit state, first datetime seen |
| `all_acquisitions.csv` | one row per acquisition, repeats included, no geometry (joins back on `grid_code`) |
| `discovery_curve.png` | unique swaths found vs granules scanned |
| `ocean_land_coastal_swaths.png` | the classified world map |
| `swaths_{ocean,coastal,land}_only.png` | one world map per class |
| `region_{REGION_NAME}_swaths.png` | swaths over one continent or country, with a satellite basemap |

Both CSVs only ever get appended to, so they're safe to read mid scan.

## 1. Configuration

Everything that changes between machines is in this one cell.

The credentials file is a few lines of plain Python, read with `exec` in the next cell, so the values need quotes:

```
maap_offline_token = "..."
maap_client_id = "..."
maap_client_secret = "..."
```

They come from a MAAP account at explorer.maap.eo.esa.int. Leaving out the client secret gives a `401 unauthorized_client`. The offline token also dies after about 90 days, or sooner if it isn't used, with no warning, so a setup that worked last month can suddenly 401 too. Never commit this file.

The two thresholds decide the classes later on: an ocean fraction of 0.95 or more is ocean, 0.05 or less is land, anything in between is coastal.

In [ ]:
from pathlib import Path  # nicer than hardcoding slashes, plays fine cross-platform too

SECRETS_PATH = Path.home() / 'eo_credentials.txt'  # where my credentials file lives (used to be secrets.txt), keep it OUT of the repo
BASE_DIR = Path.home() / 'STEP_BACK' / 'sidequest_check'  # everything this notebook writes goes under here, name's a leftover from what I was calling this side-project
COLLECTION_ID = "BiomassLevel1a"  # the STAC collection I'm scanning
KEYCLOAK_TOKEN_URL = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"  # MAAP's Keycloak token endpoint, confirmed working as of writing this
CATALOG_URL = "https://catalog.maap.eo.esa.int/catalogue/"  # MAAP's STAC catalogue root

BATCH_SIZE = 500  # granules between each checkpoint write, 500 felt like a reasonable middle ground between "checkpoint often enough" and "don't hammer the disk"

# reconnect behaviour on API errors, this matters a LOT, see the caveats in the write-up
MAX_OUTER_ATTEMPTS = 30  # how many times the scan will try to reconnect before it just gives up - I'd keep this high, it really does need to reconnect a lot over a multi-hour scan
OUTER_RECONNECT_DELAY = 30  # seconds to sit and wait before trying to reconnect again, don't want to hammer the API right after it's just fallen over

# thresholds used later on to classify each swath as ocean/coastal/land
OCEAN_THRESHOLD = 0.95  # ocean_fraction >= this -> "ocean"
LAND_ONLY_THRESHOLD = 0.05  # ocean_fraction <= this -> "land", anything in between is "coastal"

# derived paths - shouldn't need to touch anything below this line, honestly
BASE_DIR.mkdir(parents=True, exist_ok=True)  # make sure the working dir actually exists before anything tries to write into it
STATE_FILE = BASE_DIR / "scan_state.json"  # the resume checkpoint lives here
ALL_SWATHS_CSV = BASE_DIR / "all_swaths.csv"  # one row per unique swath location
ALL_ACQUISITIONS_CSV = BASE_DIR / "all_acquisitions.csv"  # one row per individual acquisition, repeats and all
OUTPUT_DIR = BASE_DIR  # plots get dumped here too, didn't see the point of a separate folder

print("Configuration set:")
print(f" SECRETS_PATH = {SECRETS_PATH}")
print(f" BASE_DIR = {BASE_DIR}")
print(f" STATE_FILE = {STATE_FILE}")
print(f" ALL_SWATHS_CSV = {ALL_SWATHS_CSV}")
print(f" ALL_ACQUISITIONS_CSV = {ALL_ACQUISITIONS_CSV}")

## 2. Setup and login

Run this at the start of every session. It swaps the offline token for a short lived access token through MAAP's Keycloak, opens the STAC catalogue, and reloads whatever has already been checkpointed, so it's safe to rerun halfway through a scan.

If `scan_state.json` has gone missing but `all_acquisitions.csv` is still there, it rebuilds the checkpoint from the CSV instead of starting from zero (this saved me once).

It also builds one big land polygon from Natural Earth 1:50 m with the lakes cut out, so inland water like the Caspian counts as water and not land. The first time it runs, cartopy downloads the shapefiles.

In [ ]:
# SETUP CELL, run this at the start of every session
exec(SECRETS_PATH.read_text())  # loads maap_offline_token, maap_client_id, maap_client_secret into scope, bit hacky but does the job

# Imports
import json  # for the state checkpoint file
import gc  # forcing a collect after every checkpoint, see below
import time  # for the reconnect delay
import requests  # for the token request itself
import pandas as pd
from shapely.geometry import shape  # turning STAC geometry dicts into real geometries
from shapely.ops import unary_union  # merging all the land polygons into one big shape
from pystac_client import Client
from pystac_client.exceptions import APIError  # the thing we specifically need to catch, see Section 3
import cartopy.io.shapereader as shpreader  # for the Natural Earth land/lake shapefiles

# Authentication (MAAP via Keycloak)
def get_access_token():
    resp = requests.post(
        KEYCLOAK_TOKEN_URL,
        data={
            "grant_type": "refresh_token",  # exchanging the long-lived offline token for a short-lived access token
            "client_id": maap_client_id,
            "client_secret": maap_client_secret,  # required, leaving this out is what gives the 401 I mentioned above
            "refresh_token": maap_offline_token,
            "scope": "offline_access openid",
        },
    )
    resp.raise_for_status()  # fail loudly here rather than limping on with no token
    return resp.json()["access_token"]

access_token = get_access_token()
print("Access token retrieved successfully.")

# STAC catalog connection
catalog = Client.open(
    CATALOG_URL,
    headers={"Authorization": f"Bearer {access_token}"},
)

# Resume state: loads scan_state.json if it's there, or rebuilds it from the CSV if the state file's gone missing but real data hasn't
if STATE_FILE.exists():
    with open(STATE_FILE) as f:
        state = json.load(f)  # just trust what's on disk
    print(f"Reloaded checkpoint: {state.get('n_scanned', 0)} granules already scanned, "
          f"{len(state.get('seen_grid_codes', []))} unique swaths, "
          f"last_datetime={state.get('last_datetime')}")

elif ALL_ACQUISITIONS_CSV.exists():
    # this saved me once when I accidentally deleted scan_state.json but the CSV was untouched
    print(f"No {STATE_FILE.name} found, but {ALL_ACQUISITIONS_CSV.name} exists"
          f"reconstructing checkpoint from existing CSV data instead of starting fresh.")

    acq_df = pd.read_csv(ALL_ACQUISITIONS_CSV)
    acq_df["datetime"] = pd.to_datetime(acq_df["datetime"])  # need real datetimes to find the max

    n_scanned = len(acq_df)  # however many rows are already logged
    seen_grid_codes = sorted(acq_df["grid_code"].dropna().unique().tolist())  # rebuild the dedup set from what's already on disk
    max_dt = acq_df["datetime"].max()  # this becomes the resume point
    last_datetime = max_dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"  # back to the STAC datetime string format
    boundary_ids = acq_df.loc[acq_df["datetime"] == max_dt, "id"].tolist()  # every id tied at that exact last datetime, so we don't double count or skip ties

    state = {
        "n_scanned": n_scanned,
        "seen_grid_codes": seen_grid_codes,
        "last_datetime": last_datetime,
        "boundary_ids": boundary_ids,
    }
    with open(STATE_FILE, "w") as f:
        json.dump(state, f)  # write it back out so next time this branch isn't needed

    print(f"Reconstructed checkpoint: {n_scanned} granules, {len(seen_grid_codes)} unique swaths, "
          f"last_datetime={last_datetime}, {len(boundary_ids)} boundary id(s)")
else:
    state = {"n_scanned": 0}  # genuinely starting from zero
    print("No checkpoint and no existing CSV data found... starting fresh.")

# Land geometry, used for the ocean-fraction masking. Natural Earth 50m, lake-aware
# (lakes subtracted out so inland seas like the Caspian read as water and not land)
land_shp = shpreader.natural_earth(resolution="50m", category="physical", name="land")
land_geoms = [shape(g) for g in shpreader.Reader(land_shp).geometries() if g is not None]  # drop any Nones, there shouldn't be any but just in case
land_raw_union = unary_union(land_geoms)  # one big land polygon instead of thousands of little ones, much faster to intersect against later
lakes_shp = shpreader.natural_earth(resolution="50m", category="physical", name="lakes")
lake_geoms = [shape(g) for g in shpreader.Reader(lakes_shp).geometries() if g is not None]
lakes_union = unary_union(lake_geoms) if lake_geoms else None
land_union = land_raw_union.difference(lakes_union) if lakes_union is not None else land_raw_union  # subtract the lakes so they don't get miscounted as land
print(f"land_union built: {len(land_geoms)} land polygons, lake-aware (lakes subtracted)")
print("\nSetup complete. catalog, COLLECTION_ID, STATE_FILE, ALL_SWATHS_CSV, "
      "ALL_ACQUISITIONS_CSV, BATCH_SIZE, land_union, state are all defined.")

## 3. Scan the collection

The core of the notebook, and the part that took the most trial and error. It goes through the collection in time order, oldest first, so "carry on from the last datetime seen" actually means something.

For every granule:

- a row goes in `all_acquisitions.csv` (cheap, no geometry), so repeat visits are counted
- if its `grid:code` hasn't been seen before, the footprint is intersected with the land polygon to get the ocean fraction, and a row goes in `all_swaths.csv`. That intersection is the expensive bit, so it only happens once per location

Every `BATCH_SIZE` granules both CSVs get the new rows appended and the checkpoint is written.

Two bugs cost real time here and are worth knowing about:

- **Server side product filter.** An earlier version asked the catalogue to filter on `product:type`. It silently dropped valid S2 and S3 SCS products with no error at all. Now everything comes back and the type is checked here, accepting anything ending in `SCS__1S`.
- **Errors read as the end.** An `APIError` in the middle of a page used to surface as a normal `StopIteration`, so a single 502 quietly ended the scan as if the collection was finished. Now any API error raises `APIReconnectNeeded`, the loop reconnects with a fresh search from the checkpoint, and only a `StopIteration` from a healthy iterator counts as done.

The datetime filter is inclusive, so the ids already handled at the exact resume datetime are remembered (`boundary_ids`) and skipped, while anything else at that same datetime still gets processed.

If it stops after `MAX_OUTER_ATTEMPTS` reconnects without reaching the end, nothing is lost, just run the cell again.

In [ ]:
# SCAN CELL, self-healing, resumable, checkpointed every BATCH_SIZE granules
class APIReconnectNeeded(Exception):
    """Raised on ANY APIError mid-scan. Signals the outer loop to reconnect
    with a fresh search() call rather than retrying the same iterator, which
    by that point isn't trustworthy anymore anyway."""
    pass

def next_or_reconnect(it):
    try:
        return next(it)  # normal path, most of the time this just works
    except StopIteration:
        raise  # a real StopIteration from a healthy iterator is genuine completion, let it through as-is
    except APIError as e:
        raise APIReconnectNeeded(str(e)) from e  # anything else means the session's broken, don't pretend otherwise

seen_grid_codes = set(state.get("seen_grid_codes", []))  # the dedup set, carried over from checkpoint if there is one
write_header = not ALL_SWATHS_CSV.exists()  # only write CSV headers the very first time the file's created
write_header_acq = not ALL_ACQUISITIONS_CSV.exists()
n_new = state["n_scanned"]  # running total across ALL sessions, not just this one
n_duplicate_skips = 0  # per-session counters, just for the printed progress line
n_wrong_product_type_skips = 0
last_seen_datetime = state.get("last_datetime")

boundary_ids = set(state.get("boundary_ids", []))  # every id already processed at the current resume datetime
if not boundary_ids and state.get("last_id"):
    boundary_ids = {state["last_id"]}  # backwards compatibility with an older checkpoint format that only stored a single last_id

session_start_scanned = n_new  # so the progress printout can show "+X this session" separately from the lifetime total
session_start_remaining = None

outer_attempt = 0
finished = False

while outer_attempt < MAX_OUTER_ATTEMPTS and not finished:
    outer_attempt += 1
    resume_datetime = state.get("last_datetime")  # re-read every loop, since it gets updated as we go
    datetime_filter = f"{resume_datetime}/.." if resume_datetime else None  # open-ended range from the resume point onward, or unbounded if starting fresh

    search = catalog.search(
        collections=[COLLECTION_ID],
        method="GET",
        datetime=datetime_filter,
        sortby="+properties.datetime",  # ascending, so "resume from the last datetime seen" is actually well-defined
        max_items=None,
    )

    try:
        remaining_now = search.matched()  # just for the progress printout, not load-bearing
    except Exception:
        remaining_now = None  # some catalog responses don't return a match count, don't let that break the scan
    if session_start_remaining is None:
        session_start_remaining = remaining_now  # only set this once, on the very first reconnect of the session

    label = f"datetime >= {resume_datetime}" if resume_datetime else "the beginning"
    print(f"[Reconnect number{outer_attempt}] Resuming from {label} , {remaining_now} granules remain")

    item_iter = search.items()  # fresh iterator, fresh HTTP session, this is the whole point of reconnecting rather than retrying
    batch_rows = []  # unique-swath rows waiting to be flushed to ALL_SWATHS_CSV
    acquisitions_batch = []  # every-acquisition rows waiting to be flushed to ALL_ACQUISITIONS_CSV
    page_error = False

    try:
        while True:
            try:
                item = next_or_reconnect(item_iter)  # pulls the next STAC item, or raises if something's gone wrong
            except StopIteration:
                finished = True  # a real, clean end of the collection
                break
            except APIReconnectNeeded as e:
                print(f"\n  API error requires reconnect: {e}")
                page_error = True  # break out to the outer while loop, which will reconnect and carry on from the checkpoint
                break

            item_dt = item.properties.get("datetime")

            # the datetime filter is inclusive, so at the exact resume boundary we only skip ids we've already processed
            # anything tied at that same datetime that we haven't reached yet is NOT in boundary_ids and still gets handled normally
            if item_dt == resume_datetime and item.id in boundary_ids:
                continue

            # client-side product type check, accepts any SCS product (S1/S2/S3/whatever comes next)
            # rather than trusting a server-side filter, see the write-up for why that bit me before
            product_type = item.properties.get("product:type", "")
            if not product_type.endswith("SCS__1S"):
                n_wrong_product_type_skips += 1
                continue

            n_new += 1
            if item_dt != last_seen_datetime:
                boundary_ids = set()  # new datetime, so the old boundary_ids no longer apply
            last_seen_datetime = item_dt
            boundary_ids.add(item.id)  # track this id in case we get interrupted at exactly this datetime

            grid_code = item.properties.get("grid:code")  # the actual swath location identifier

            # log EVERY acquisition including repeats, lightweight, no geometry
            # geometry's essentially fixed per grid_code anyway so it gets joined back in from ALL_SWATHS_CSV at plot time
            acquisitions_batch.append({
                "id": item.id,
                "grid_code": grid_code,
                "datatake_id": item.properties.get("eopf:datatake_id"),
                "absolute_orbit": item.properties.get("sat:absolute_orbit"),
                "datetime": item_dt,
                "orbit_state": item.properties.get("sat:orbit_state"),
                "product_type": product_type,
            })

            if grid_code is not None and grid_code in seen_grid_codes:
                n_duplicate_skips += 1  # already got this swath, just counting it, don't redo the expensive geometry work
            else:
                if grid_code is not None:
                    seen_grid_codes.add(grid_code)  # mark it seen before doing anything else with it
                try:
                    footprint = shape(item.geometry)  # STAC geometry dict -> shapely geometry
                    if footprint.area > 0:
                        land_overlap = footprint.intersection(land_union)  # the expensive bit, only done once per unique swath
                        ocean_fraction = 1 - (land_overlap.area / footprint.area)
                    else:
                        ocean_fraction = None  # degenerate footprint, nothing sensible to compute
                except Exception:
                    ocean_fraction = None  # don't let one weird geometry take the whole scan down

                # save every unique swath's fraction + geometry regardless of value
                # thresholds get applied later at plot time, so nothing's thrown away at this stage
                if ocean_fraction is not None:
                    batch_rows.append({
                        "id": item.id,
                        "grid_code": grid_code,
                        "geometry_wkt": footprint.wkt,  # storing as WKT text, easiest thing to round-trip through a CSV
                        "ocean_fraction": ocean_fraction,
                        "orbit_state": item.properties.get("sat:orbit_state"),
                        "datetime": item_dt,
                        "product_type": product_type,
                    })

            if n_new % BATCH_SIZE == 0:  # checkpoint time
                if batch_rows:
                    pd.DataFrame(batch_rows).to_csv(ALL_SWATHS_CSV, mode="a", header=write_header, index=False)  # append, not overwrite
                    write_header = False  # header only ever gets written once, right at the very start
                    batch_rows = []
                if acquisitions_batch:
                    pd.DataFrame(acquisitions_batch).to_csv(ALL_ACQUISITIONS_CSV, mode="a", header=write_header_acq, index=False)
                    write_header_acq = False
                    acquisitions_batch = []
                state["n_scanned"] = n_new
                state["seen_grid_codes"] = list(seen_grid_codes)  # sets aren't JSON serialisable, back to a list for the checkpoint file
                state["last_datetime"] = last_seen_datetime
                state["boundary_ids"] = list(boundary_ids)
                state.pop("last_id", None)  # drop the old single-id field if it's still hanging around from an older checkpoint
                with open(STATE_FILE, "w") as f:
                    json.dump(state, f)
                gc.collect()  # a multi-hour scan builds up a lot of garbage otherwise, this just keeps memory sane
                session_progress = n_new - session_start_scanned
                pct = f"{100 * session_progress / session_start_remaining:.1f}%" if session_start_remaining else "?"
                print(f"  scanned {n_new} total (+{session_progress} this session, ~{pct} of remaining) "
                      f"| {len(seen_grid_codes)} unique swaths | {n_duplicate_skips} dupes skipped "
                      f"| {n_wrong_product_type_skips} non-SCS skipped")
    finally:
        # flush whatever's left in the batch even if we broke out early, don't want to lose a partial batch on an error
        if batch_rows:
            pd.DataFrame(batch_rows).to_csv(ALL_SWATHS_CSV, mode="a", header=write_header, index=False)
            write_header = False
        if acquisitions_batch:
            pd.DataFrame(acquisitions_batch).to_csv(ALL_ACQUISITIONS_CSV, mode="a", header=write_header_acq, index=False)
            write_header_acq = False
        state["n_scanned"] = n_new
        state["seen_grid_codes"] = list(seen_grid_codes)
        state["last_datetime"] = last_seen_datetime
        state["boundary_ids"] = list(boundary_ids)
        state.pop("last_id", None)
        with open(STATE_FILE, "w") as f:
            json.dump(state, f)  # always leave a checkpoint behind, whatever happened above

    if page_error and not finished:
        print(f"Reconnecting in {OUTER_RECONNECT_DELAY}s (attempt {outer_attempt}/{MAX_OUTER_ATTEMPTS})...")
        time.sleep(OUTER_RECONNECT_DELAY)  # give the API a breather before hammering it again

if finished:
    print(f"\nDone! Reached the end of the collection.")
else:
    print(f"\nStopped after {outer_attempt} reconnect attempts without reaching the end. "
          f"Progress is saved, just rerun this cell to keep going from here.")
print(f"Scanned {n_new} granules total ({n_duplicate_skips} were repeat visits to an already-seen swath).")
print(f"{n_wrong_product_type_skips} non-SCS items skipped client-side.")
print(f"{len(seen_grid_codes)} unique swath locations logged (all fractions). Results in {ALL_SWATHS_CSV}")
print(f"Raw per-acquisition log (every granule, with repeats) in {ALL_ACQUISITIONS_CSV}")

## Reconciling the checkpoint (only if needed)

Only for when the live "unique swaths" count looks lower than what's actually in `all_swaths.csv`, which can happen if `scan_state.json` was rebuilt from a partial CSV at some point. It merges the two sets of grid codes. Don't run it while the scan cell is going, it'll just be overwritten at the next checkpoint. Rerun Section 2 afterwards.

In [ ]:
# reconcile scan_state.json's seen_grid_codes with the full accumulated all_swaths.csv
full_swaths_df = pd.read_csv(ALL_SWATHS_CSV)
csv_grid_codes = set(full_swaths_df["grid_code"].dropna().unique())  # the source of truth, really
with open(STATE_FILE) as f:
    current_state = json.load(f)
state_grid_codes = set(current_state.get("seen_grid_codes", []))
merged = state_grid_codes | csv_grid_codes  # just take the union, can't lose anything that way
print(f"State file had: {len(state_grid_codes)} grid codes")
print(f"CSV has: {len(csv_grid_codes)} grid codes")
print(f"Merged total:{len(merged)} grid codes")
current_state["seen_grid_codes"] = sorted(merged)
with open(STATE_FILE, "w") as f:
    json.dump(current_state, f)
print("Saved, rerun Section 2 (Setup) to reload this into `state` before continuing the scan.")

## Diagnostics: date coverage and product types

How far through the mission timeline the scan has got, and a check that S2 and S3 products are actually turning up and not just S1.

In [ ]:
# date coverage and product type breakdown
acq_df = pd.read_csv(ALL_ACQUISITIONS_CSV)
acq_df["datetime"] = pd.to_datetime(acq_df["datetime"])
earliest = acq_df["datetime"].min()
latest = acq_df["datetime"].max()
span = latest - earliest
print(f"{len(acq_df)} acquisitions logged so far")
print(f"Earliest acquisition scanned: {earliest}")
print(f"Latest acquisition scanned: {latest}")
print(f"Date range covered so far: {span.days} days")
if "product_type" in acq_df.columns:  # older CSVs from before I added this column wouldn't have it
    print("\n Product type breakdown:")
    print(acq_df["product_type"].value_counts())

## Diagnostics: is the catalogue actually representative?

The one that matters most. Discovery isn't a smooth curve that levels off. It's long flat stretches of nothing but repeat visits, broken by sudden bursts of new swaths, and there's no telling where in the timeline those bursts land. In my run about 11 % of the final unique total showed up in only the last 15 % of the scan. So don't trust a partial scan just because the curve has looked flat for a while.

In [ ]:
# discovery curve: cumulative unique swaths vs cumulative scanned
import matplotlib.pyplot as plt
from datetime import datetime  # needed for the title timestamp below (easy to forget this import if you're just copy-pasting cells around, ask me how I know)

acq_df = pd.read_csv(ALL_ACQUISITIONS_CSV)
acq_df["datetime"] = pd.to_datetime(acq_df["datetime"])
acq_df = acq_df.sort_values("datetime").reset_index(drop=True)  # has to be chronological for "cumulative" to mean anything
acq_df["is_first_occurrence"] = ~acq_df["grid_code"].duplicated()  # True the first time a grid_code shows up, False every time after
acq_df["cumulative_unique"] = acq_df["is_first_occurrence"].cumsum()  # running total of unique swaths
acq_df["cumulative_scanned"] = range(1, len(acq_df) + 1)  # just the row index, offset by 1

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(acq_df["cumulative_scanned"], acq_df["cumulative_unique"], linewidth=1.2)
ax.set_xlabel("Items scanned")
ax.set_ylabel("Unique swaths discovered")
ax.set_title(
    f"BIOMASS L1A SLC new product paths discovered as of {datetime.now():%d-%m-%y}\n"
    f"Note: Flat curves = repeat aquisition phase | Positive slope = New paths discovered"
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "discovery_curve.png", dpi=150, bbox_inches="tight")
plt.show()

N = min(200_0000, len(acq_df))  # last 2 million rows or the whole thing if there's less than that, just a "recent window" sanity check
recent = acq_df.tail(N)
new_in_recent = recent["is_first_occurrence"].sum()
print(f"In the most recent {N} scanned items: {new_in_recent} were new unique swaths "
      f"({100 * new_in_recent / N:.2f}%)")

## World map: ocean, coastal and land

Reads `all_swaths.csv` fresh, so it works any time, mid scan included. Every unique swath is classed with the thresholds from Section 1 and drawn on one world map: blue ocean, green land, red coastal (drawn last so they don't get buried). Natural Earth 1:50 m coastlines instead of the default 1:110 m, which matters most around Antarctica.

Footprints that cross the antimeridian are split where they jump, so they don't draw a line straight across the map.

One caveat. ESA says the Biomass radar is switched off within line of sight of the US Space Object Tracking Radar stations, which "mainly excludes North America and Europe", but that's only published as a picture, not coordinates. So the gap over those continents is real, but any boundary I draw for it is a rough rectangle, not the true line of sight.

In [ ]:
# ocean/land/coastal world map
from shapely import wkt as shapely_wkt
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

all_df = pd.read_csv(ALL_SWATHS_CSV)
swaths_geo = all_df.drop_duplicates(subset="grid_code").copy()  # belt and braces, all_swaths.csv should already be unique by grid_code but just in case
swaths_geo["geometry"] = swaths_geo["geometry_wkt"].apply(shapely_wkt.loads)  # WKT text back into real shapely geometries
print(f"{len(swaths_geo)} unique swath locations loaded")

ocean_df = swaths_geo[swaths_geo["ocean_fraction"] >= OCEAN_THRESHOLD]
coastal_df = swaths_geo[(swaths_geo["ocean_fraction"] > LAND_ONLY_THRESHOLD) & (swaths_geo["ocean_fraction"] < OCEAN_THRESHOLD)]
land_df = swaths_geo[swaths_geo["ocean_fraction"] <= LAND_ONLY_THRESHOLD]

print(f" {len(ocean_df)} ocean (>= {OCEAN_THRESHOLD})")
print(f" {len(coastal_df)} coastal ({LAND_ONLY_THRESHOLD} < fraction < {OCEAN_THRESHOLD})")
print(f" {len(land_df)} land (<= {LAND_ONLY_THRESHOLD})")

def plot_ring_dateline_safe(ax, xs, ys, **kwargs):
    """Plots a polygon ring, splitting it wherever the path jumps across the
    antimeridian, so it doesn't draw a spurious line straight across the map."""
    xs = list(xs); ys = list(ys)
    if len(xs) < 2:
        return
    jumps = [i for i in range(1, len(xs)) if abs(xs[i] - xs[i - 1]) > 180]  # a >180 degree jump between consecutive points means we crossed the seam
    segments = []
    start = 0
    for j in jumps:
        segments.append((xs[start:j], ys[start:j]))  # everything up to the jump is one segment
        start = j
    segments.append((xs[start:], ys[start:]))  # and whatever's left after the last jump
    for seg_x, seg_y in segments:
        if len(seg_x) >= 2:
            ax.plot(seg_x, seg_y, transform=ccrs.PlateCarree(), **kwargs)

fig = plt.figure(figsize=(12, 20))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -81, 90], crs=ccrs.PlateCarree())  # crops the far south a bit, otherwise the dense tangle of Antarctic swaths dominates the whole plot

# 50m resolution, not the coarse 110m default, matters most around Antarctica
land_50m = cfeature.NaturalEarthFeature('physical', 'land', '50m',
                                          facecolor="#b8e0b8", edgecolor="none")
ocean_50m = cfeature.NaturalEarthFeature('physical', 'ocean', '50m',
                                           facecolor="#c9e3f0", edgecolor="none")
ax.add_feature(ocean_50m, zorder=0)
ax.add_feature(land_50m, zorder=0)

# draw order matters here: land first, then ocean, then coastal on top, so coastal swaths don't get buried
for geom in land_df["geometry"]:
    geoms = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)  # handle MultiPolygons the same way as single Polygons
    for g in geoms:
        xs, ys = g.exterior.xy
        plot_ring_dateline_safe(ax, xs, ys, color="#1a5c1a", linewidth=0.6, alpha=0.7)
for geom in ocean_df["geometry"]:
    geoms = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for g in geoms:
        xs, ys = g.exterior.xy
        plot_ring_dateline_safe(ax, xs, ys, color="#0a1a5c", linewidth=0.6, alpha=0.7)
for geom in coastal_df["geometry"]:
    geoms = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for g in geoms:
        xs, ys = g.exterior.xy
        plot_ring_dateline_safe(ax, xs, ys, color="#d62728", linewidth=0.7, alpha=0.8)  # slightly thicker/more opaque so coastal swaths stand out on top

from datetime import datetime

ax.set_title(
    f"BIOMASS L1A SLC path locations available as of {datetime.now():%d-%m-%y}\n"
    f"Ocean (blue): {len(ocean_df)}  |  Land (green): {len(land_df)}  |  Coastal (red): {len(coastal_df)}"
)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ocean_land_coastal_swaths.png", dpi=150, bbox_inches="tight")
plt.show()

## The same thing as three separate maps

One map per class, which is honestly easier to read than three overlapping colours on one busy plot.

In [ ]:
# plot ocean / coastal / land swaths as three SEPARATE maps rather than one combined overlay
from shapely import wkt as shapely_wkt
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_ring_dateline_safe(ax, xs, ys, **kwargs):  # same helper as before, redefined here so this cell can run standalone
    xs = list(xs); ys = list(ys)
    if len(xs) < 2:
        return
    jumps = [i for i in range(1, len(xs)) if abs(xs[i] - xs[i - 1]) > 180]
    segments = []
    start = 0
    for j in jumps:
        segments.append((xs[start:j], ys[start:j]))
        start = j
    segments.append((xs[start:], ys[start:]))
    for seg_x, seg_y in segments:
        if len(seg_x) >= 2:
            ax.plot(seg_x, seg_y, transform=ccrs.PlateCarree(), **kwargs)

all_df = pd.read_csv(ALL_SWATHS_CSV)
swaths_geo = all_df.drop_duplicates(subset="grid_code").copy()
swaths_geo["geometry"] = swaths_geo["geometry_wkt"].apply(shapely_wkt.loads)
ocean_df = swaths_geo[swaths_geo["ocean_fraction"] >= OCEAN_THRESHOLD]
coastal_df = swaths_geo[(swaths_geo["ocean_fraction"] > LAND_ONLY_THRESHOLD) & (swaths_geo["ocean_fraction"] < OCEAN_THRESHOLD)]
land_df = swaths_geo[swaths_geo["ocean_fraction"] <= LAND_ONLY_THRESHOLD]

CATEGORIES = [
    ("ocean",   ocean_df,   "#0a1a5c"),
    ("coastal", coastal_df, "#d62728"),
    ("land",    land_df,    "#1a5c1a"),
]  # driving the loop below off this list so I'm not copy-pasting the same plot three times

for label, df, color in CATEGORIES:
    fig = plt.figure(figsize=(12, 9))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([-180, 180, -81, 90], crs=ccrs.PlateCarree())

    land_50m = cfeature.NaturalEarthFeature('physical', 'land', '50m', facecolor="#b8e0b8", edgecolor="none")
    ocean_50m = cfeature.NaturalEarthFeature('physical', 'ocean', '50m', facecolor="#c9e3f0", edgecolor="none")
    ax.add_feature(ocean_50m, zorder=0)
    ax.add_feature(land_50m, zorder=0)

    for geom in df["geometry"]:
        geoms = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
        for g in geoms:
            xs, ys = g.exterior.xy
            plot_ring_dateline_safe(ax, xs, ys, color=color, linewidth=0.7, alpha=0.8)

    ax.set_title(f"BIOMASS L1a SLC product path locations with {label} ({len(df)} swaths)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"swaths_{label}_only.png", dpi=150, bbox_inches="tight")
    plt.show()

print(f"{len(ocean_df)} ocean / {len(coastal_df)} coastal / {len(land_df)} land, 3 maps saved to {OUTPUT_DIR}")

## Zooming in on a continent or country

Picks out the swaths over one region and plots them twice side by side: on a simple map, and in white on top of satellite imagery. Mostly a quick "did we actually get anything here?" check. Change `REGION_NAME` to any key in the boxes below.

The boxes are rough rectangles, not real borders, so the corners catch bits of neighbours (the Africa box clips the Arabian Peninsula, for example). Countries are coastal ones only, since landlocked countries don't matter for ocean work, and most of Europe and North America are left out because of the radar restriction above.

Two fixes are in this version:

- **The antimeridian, properly this time.** A footprint crossing the dateline is stored with longitudes jumping from about +179 to about -179. Shapely doesn't know the Earth wraps around, so it reads that as a shape stretching across almost the whole map. That broke the `intersects()` test that picks swaths for a region as well as the plot. The geometry is now unwrapped into continuous longitudes, plus copies shifted by 360 either way, before the test, not only when drawing.
- **The width check.** A real swath is a strip a few hundred km wide, so anything far wider is a corrupted footprint. A cutoff in degrees of longitude is wrong near the poles, where a degree covers far fewer km, and it was throwing out perfectly good Antarctic swaths. Converting to km with each swath's own latitude fixes it everywhere at once.

The satellite panel downloads map tiles when it runs, so it needs internet. `ZOOM_LEVEL` trades detail against time: about 4 to 5 for a continent, 6 to 7 for a country.

In [ ]:
from shapely import wkt as shapely_wkt
from shapely.geometry import box, Polygon, MultiPolygon
from shapely.affinity import translate
from shapely.ops import unary_union
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.img_tiles import GoogleTiles  # for the real satellite imagery panel

REGION_BOXES = {
    "Africa":        box(-20, -35, 55, 38),
    "South_America": box(-82, -56, -34, 13),
    "North_America": box(-170, 5, -50, 75),  # limited products here due to the SOTR restriction, see the caveat above
    "Europe":        box(-25, 35, 40, 72),   # same restriction applies here
    "Asia":          box(40, -10, 180, 78),
    "Australia":     box(112, -45, 155, -10),
    "Antarctica":    box(-180, -90, 180, -60),
}
# COUNTRY_BOXES below covers coastal countries with ocean/SSS relevance across every continent
# EXCEPT Europe (restricted) and most of North America (restricted, except the far south/Caribbean
# which already sits outside the restricted box above). Landlocked countries are deliberately left
# out, no coastline means no relevance to ocean/coastal validation. Same caveat as always: these are
# rectangles, not real borders.

COUNTRY_BOXES = {
    # Central America and Caribbean (the far-south bit of North America that isn't restricted)
    "Mexico":            box(-118, 14, -86, 33),
    "Guatemala":         box(-92.5, 13.5, -88, 17.9),
    "Belize":            box(-89.3, 15.9, -88, 18.5),
    "Honduras":          box(-89.4, 12.9, -83, 16.5),
    "El_Salvador":       box(-90.1, 13.1, -87.7, 14.5),
    "Nicaragua":         box(-87.7, 10.7, -83, 15),
    "Costa_Rica":        box(-86, 8, -82.5, 11.2),
    "Panama":            box(-83, 7, -77, 9.7),
    "Cuba":              box(-85, 19.8, -74, 23.3),
    "Dominican_Republic": box(-72.1, 17.5, -68.3, 19.9),
    "Jamaica":           box(-78.4, 17.6, -76.1, 18.6),
    "Bahamas":           box(-79, 20.9, -72.7, 27.3),

    # South America
    "Brazil":            box(-74, -34, -34, 6),
    "Argentina":         box(-74, -55, -53, -22),
    "Chile":             box(-76, -56, -66, -17),
    "Peru":              box(-82, -18, -68, 0),
    "Colombia":          box(-79, -4, -66, 13),
    "Venezuela":         box(-73, 0, -59, 12),
    "Ecuador":           box(-81, -5, -75, 1.5),
    "Uruguay":           box(-58, -35, -53, -30),
    "Guyana":            box(-61, 1, -56, 8.5),
    "Suriname":          box(-58, 1.8, -54, 6),

    # Africa
    "Morocco":           box(-17.1, 27.6, -1, 35.9),
    "Algeria":           box(-8.7, 18.9, 12, 37.1),
    "Tunisia":           box(7.5, 30.2, 11.6, 37.6),
    "Libya":             box(9.3, 19.5, 25.2, 33.2),
    "Egypt":             box(25, 22, 37, 32),
    "Sudan":             box(21.8, 8.7, 38.6, 22),
    "Eritrea":           box(36.4, 12.4, 43.1, 18),
    "Djibouti":          box(41.7, 10.9, 43.4, 12.7),
    "Somalia":           box(40.9, -1.7, 51.4, 12),
    "Kenya":             box(34, -5, 42, 5),
    "Tanzania":          box(29.3, -11.7, 40.4, -1),
    "Mozambique":        box(30.2, -26.9, 40.8, -10.5),
    "South_Africa":      box(16, -35, 33, -22),
    "Namibia":           box(11.7, -28.9, 25.3, -16.9),
    "Angola":            box(11.7, -18, 24.1, -4.4),
    "DR_Congo":          box(12.2, -13.5, 31.3, 5.4),  # small coastline near the Congo river mouth, easy to forget this one's coastal at all
    "Congo":             box(11.1, -5, 18.6, 3.7),
    "Gabon":             box(8.7, -3.9, 14.5, 2.3),
    "Cameroon":          box(8.5, 1.7, 16.2, 13.1),
    "Nigeria":           box(2, 4, 15, 14),
    "Ghana":             box(-3.3, 4.7, 1.2, 11.2),
    "Ivory_Coast":       box(-8.6, 4.3, -2.5, 10.7),
    "Senegal":           box(-17.6, 12.3, -11.3, 16.7),
    "Mauritania":        box(-17.1, 14.7, -4.8, 27.3),
    "Madagascar":        box(43.2, -25.6, 50.5, -11.9),

    # Asia / Middle East
    "Russia":            box(19, 41, 180, 82),  # touches the antimeridian out in Chukotka, same seam issue as the Asia box below
    "China":             box(73, 18, 135, 54),
    "North_Korea":       box(124.3, 37.7, 130.7, 43),
    "South_Korea":       box(125, 33, 130, 39),
    "Japan":             box(129, 31, 146, 46),
    "India":             box(68, 6, 97, 36),
    "Pakistan":          box(60.9, 23.6, 77.8, 37.1),
    "Bangladesh":        box(88, 20.7, 92.7, 26.6),
    "Myanmar":           box(92.2, 9.8, 101.2, 28.5),
    "Thailand":          box(97.3, 5.6, 105.6, 20.5),
    "Vietnam":           box(102.1, 8.6, 109.5, 23.4),
    "Malaysia":          box(99.6, 0.9, 119.3, 7.4),
    "Philippines":       box(116.9, 4.6, 126.6, 21.1),
    "Indonesia":         box(95, -11, 141, 6),
    "Saudi_Arabia":      box(34, 16, 56, 33),
    "UAE":               box(51, 22.6, 56.4, 26.1),
    "Iran":              box(44, 25, 63.3, 39.8),
    "Oman":              box(52, 16.6, 59.8, 26.4),
    "Yemen":             box(42.5, 12.1, 54, 19),
    "Sri_Lanka":         box(79.5, 5.9, 81.9, 9.9),
    "Turkey":            box(26, 36, 45, 42),

    # Oceania
    "Australia":         box(112, -45, 155, -10),
    "New_Zealand":       box(166, -47, 179, -34),
    "Papua_New_Guinea":  box(140.8, -11.7, 155.6, -1),
    "Fiji":              box(177, -21, 180, -12),  # simplified, Fiji's easternmost islands actually cross the antimeridian, this box only gets the western part
}

REGION_BOXES.update(COUNTRY_BOXES)  # merge so REGION_NAME below can pick either a continent or an individual country by name

###############################################################################################################
# change this to whatever you want plotted, any key from REGION_BOXES above (continent or country)
REGION_NAME = "Guyana"
###############################################################################################################

REGION_GEOM = REGION_BOXES[REGION_NAME]  # or just set this manually to a custom shapely geometry if none of the boxes above fit

# the default map projection is centred on longitude 0, seam at +-180. Any region box that touches or
# gets near 180 (Asia's box runs right up to it) makes set_extent ambiguous about which way round to
# show, and in practice it tends to fall back to showing nearly the whole globe. Fix is to recentre the
# axes' own projection away from the region, so the seam lands somewhere empty instead of through the
# middle of what I'm actually trying to look at, cartopy reprojects the raw points correctly on its own
REGION_CENTRAL_LON = {
    "Asia": 110,  # roughly the centre of Asia's box, pushes the seam out to 70 which is nowhere near any actual swaths
}

# satellite basemap panel, shown side by side with the schematic map below
# GoogleTiles fetches real imagery tiles at render time so this needs an internet connection when the
# cell runs. ZOOM_LEVEL trades detail against fetch time: ~4-5 for a full continent, ~6-7 for a single
# country, ~8+ for something small, higher is slower since it's downloading more tiles
ZOOM_LEVEL = 6
SATELLITE_TILER = GoogleTiles(style="satellite")

# antimeridian repair - root cause: a footprint genuinely crossing the dateline gets stored as raw
# lon/lat WKT jumping from ~+179 to ~-179. Shapely has no concept of the sphere wrapping around, so it
# reads that as a polygon stretching almost the full width of the map. That corrupted shape then breaks
# TWO things downstream: the .intersects() test used to build region_df (spuriously overlapping boxes
# it has no business touching), and the plot itself (draws as a huge smear). Fixing it only at plot
# time, like I originally did, can't undo a selection that already went wrong, so this fixes the
# geometry itself before either step gets anywhere near it

def unwrap_lons(xs):
    xs = list(xs)
    for i in range(1, len(xs)):
        while xs[i] - xs[i - 1] > 180:  # jumped too far east, pull it back around
            xs[i] -= 360
        while xs[i] - xs[i - 1] < -180:  # jumped too far west, same idea the other way
            xs[i] += 360
    return xs

def _unwrap_ring(coords):
    xs, ys = zip(*coords)
    return list(zip(unwrap_lons(list(xs)), ys))

def fix_polygon_dateline(poly):
    """Re-expresses a polygon's rings in a continuous (possibly beyond +-180)
    coordinate space so its true shape is preserved instead of jumping across
    the map. Can produce a polygon that extends outside [-180, 180], that's
    expected, it gets handled by the duplicate-and-shift step below."""
    ext = _unwrap_ring(list(poly.exterior.coords))
    interiors = [_unwrap_ring(list(r.coords)) for r in poly.interiors]
    try:
        return Polygon(ext, interiors)
    except Exception:
        return poly  # fall back to the original if something genuinely degenerate happens

def fix_geom_dateline(geom):
    if geom.geom_type == "Polygon":
        return fix_polygon_dateline(geom)
    elif geom.geom_type == "MultiPolygon":
        return MultiPolygon([fix_polygon_dateline(p) for p in geom.geoms])
    return geom  # anything else, just pass it through unchanged

def dateline_safe_geom(geom):
    """After unwrapping, the polygon might now live partly outside [-180, 180]
    (say, spanning 170 to 190). Region boxes are defined in the standard
    [-180, 180] space, so union in copies shifted by +-360, guarantees an
    accurate .intersects() test no matter which direction the data wrapped,
    without needing to know that in advance."""
    fixed = fix_geom_dateline(geom)
    return unary_union([fixed, translate(fixed, xoff=360), translate(fixed, xoff=-360)])

all_df = pd.read_csv(ALL_SWATHS_CSV)
swaths_geo = all_df.drop_duplicates(subset="grid_code").copy()
swaths_geo["geometry"] = swaths_geo["geometry_wkt"].apply(shapely_wkt.loads)

# selection now goes through the dateline-safe geometry instead of the raw one
region_df = swaths_geo[swaths_geo["geometry"].apply(lambda g: dateline_safe_geom(g).intersects(REGION_GEOM))].copy()

print(f"{len(swaths_geo)} unique swaths total")
print(f"{len(region_df)} swaths intersect '{REGION_NAME}'")
if len(region_df):
    print(f"  ocean fraction range: {region_df['ocean_fraction'].min():.2f} - {region_df['ocean_fraction'].max():.2f}")
    print(f"  orbit states: {region_df['orbit_state'].value_counts().to_dict()}")
    print(f"  date range: {region_df['datetime'].min()} to {region_df['datetime'].max()}")

if ALL_ACQUISITIONS_CSV.exists():
    acq_df = pd.read_csv(ALL_ACQUISITIONS_CSV)
    region_grid_codes = set(region_df["grid_code"])
    region_acq_df = acq_df[acq_df["grid_code"].isin(region_grid_codes)]
    print(f"{len(region_acq_df)} total acquisitions (incl. repeats) across those {len(region_grid_codes)} swath locations")

# drop degenerate footprints - a real BIOMASS swath is a narrow strip, only a few hundred km wide
# physically. A fixed DEGREE threshold on longitude span is wrong globally though: meridians converge
# toward the poles, so the same physical width covers far more degrees of longitude near Antarctica than
# near the equator. A raw degree cutoff correctly caught Asia's corrupted footprints but wrongly flagged
# Antarctica's perfectly normal ones, converting to actual physical width in km using each swath's own
# latitude fixes this everywhere at once instead of needing a special case for the poles
import math

MAX_REASONABLE_WIDTH_KM = 1200  # generous upper bound for what a real swath's width should ever be

def lon_span_km(geom):
    minx, miny, maxx, maxy = geom.bounds
    lon_span_deg = maxx - minx
    mean_lat = (miny + maxy) / 2  # use the middle latitude of the footprint for the conversion factor
    km_per_deg_lon = 111.32 * math.cos(math.radians(mean_lat))  # standard degrees-to-km conversion, shrinks toward the poles
    return lon_span_deg * km_per_deg_lon

region_df["width_km"] = region_df["geometry"].apply(lon_span_km)
suspect = region_df[region_df["width_km"] > MAX_REASONABLE_WIDTH_KM]
if len(suspect):
    print(f"\nExcluding {len(suspect)} swath(s) with implausible width "
          f"(>{MAX_REASONABLE_WIDTH_KM} km), likely corrupted/fallback footprints:")
    print(suspect[["id", "width_km"]].to_string(index=False))
    region_df = region_df[region_df["width_km"] <= MAX_REASONABLE_WIDTH_KM].copy()

if len(region_df):
    is_antarctica = REGION_NAME == "Antarctica"

    # side-by-side layout: left panel is the schematic map, right panel is a new satellite-imagery
    # panel sharing the same projection and extent, with the same swaths drawn on top of it
    if is_antarctica:
        proj = ccrs.SouthPolarStereo()  # PlateCarree gets pretty useless right at the pole
    else:
        central_lon = REGION_CENTRAL_LON.get(REGION_NAME, 0)
        proj = ccrs.PlateCarree(central_longitude=central_lon)

    fig = plt.figure(figsize=(20, 10))
    ax = fig.add_subplot(1, 2, 1, projection=proj)
    ax_sat = fig.add_subplot(1, 2, 2, projection=proj)

    if is_antarctica:
        ax.set_extent([-180, 180, -90, -60], crs=ccrs.PlateCarree())
        ax.gridlines(draw_labels=False, linestyle=":", alpha=0.4)
        ax_sat.set_extent([-180, 180, -90, -60], crs=ccrs.PlateCarree())
        ax_sat.gridlines(draw_labels=False, linestyle=":", alpha=0.4)
    else:
        minx, miny, maxx, maxy = REGION_GEOM.bounds
        pad = 5  # a bit of padding around the box so swaths right at the edge don't get clipped
        # extent's still expressed in standard (central_longitude=0) coordinates, cartopy transforms
        # it into the recentred axes correctly via the crs= argument, no manual math needed here
        ax.set_extent([minx - pad, maxx + pad, miny - pad, maxy + pad], crs=ccrs.PlateCarree())
        ax_sat.set_extent([minx - pad, maxx + pad, miny - pad, maxy + pad], crs=ccrs.PlateCarree())

    land_50m = cfeature.NaturalEarthFeature('physical', 'land', '50m', facecolor="#b8e0b8", edgecolor="none")
    ocean_50m = cfeature.NaturalEarthFeature('physical', 'ocean', '50m', facecolor="#c9e3f0", edgecolor="none")
    ax.add_feature(ocean_50m, zorder=0)
    ax.add_feature(land_50m, zorder=0)

    # real satellite/terrain imagery for the right-hand panel, Google-Earth-style
    ax_sat.add_image(SATELLITE_TILER, ZOOM_LEVEL)

    for geom, frac in zip(region_df["geometry"], region_df["ocean_fraction"]):
        color = "#0a1a5c" if frac >= OCEAN_THRESHOLD else ("#d62728" if frac > LAND_ONLY_THRESHOLD else "#1a5c1a")
        # plotting the RAW geometry here, with the axes recentred away from the seam, cartopy's
        # transform correctly reprojects antimeridian-crossing points on its own, no manual unwrapping
        # needed just for display (fix_geom_dateline above is only needed for the .intersects() step,
        # since shapely itself has no notion of the sphere wrapping)
        geoms = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
        for g in geoms:
            xs, ys = g.exterior.xy
            ax.plot(list(xs), list(ys), transform=ccrs.PlateCarree(), color=color, linewidth=0.7, alpha=0.8)
            # same outline drawn again on the satellite panel, white instead of the dark schematic
            # colours since dark blue/green just disappear against real imagery
            ax_sat.plot(list(xs), list(ys), transform=ccrs.PlateCarree(), color="white", linewidth=0.9, alpha=0.9)

    ax.set_title(f"BIOMASS swath coverage over {REGION_NAME} ({len(region_df)} swaths) \n [Blue = Ocean, Red = Coastal, Green = Land]")
    ax_sat.set_title(f"{REGION_NAME} satellite basemap")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"region_{REGION_NAME}_swaths.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print(f"No swaths found intersecting '{REGION_NAME}', nothing to plot.")